# PKH Context Sniper — Colab launcher

Запуск ингеста и поиска по архиву чатов Google AI Studio.

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`.

## 0. Bootstrap

Одна ячейка — монтирование, код, зависимости, конфиг, инициализация снайпера.
Запускай после каждого рестарта рантайма. Справа видно форму для настройки — меняй значения там, а не в коде.

In [ ]:
# @title ⚙️ PKH Bootstrap { display-mode: "form" }
# @markdown ### Пути (относительно `/content/drive/MyDrive/`)
CHATS_SUBPATH = "Google AI Studio"  # @param {type:"string"}
DB_SUBPATH = "pkh_chroma"  # @param {type:"string"}

# @markdown ### Модель и режим
EMBEDDING_MODEL = "bge-m3"  # @param ["bge-m3", "qwen3", "e5-instruct"]
THOUGHT_MODE = "SMART"  # @param ["OFF", "ON", "SMART"]

# @markdown ### Репозиторий
REPO_URL = "https://github.com/LilSheva/PKH.git"  # @param {type:"string"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Clone / update repo ---
import os, sys
REPO_DIR = '/content/PKH'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# --- Dependencies ---
pip_flags = '--force-reinstall' if FORCE_REINSTALL else '-q'
!pip install {pip_flags} -r {REPO_DIR}/requirements.txt
!pip install -q tqdm

# --- GPU check ---
import torch
if torch.cuda.is_available():
    print(f'✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
else:
    print('⚠️ NO GPU — Ingest will be slow. Enable Runtime → Change runtime type → T4 GPU')

# --- Config ---
from pathlib import Path
import config

config.CHATS_DIR = Path(f'/content/drive/MyDrive/{CHATS_SUBPATH}')
config.DB_DIR = Path(f'/content/drive/MyDrive/{DB_SUBPATH}')
config.MANIFEST_PATH = config.DB_DIR / 'manifest.json'
config.EMBEDDING_MODEL = EMBEDDING_MODEL
config.THOUGHT_MODE = THOUGHT_MODE

# --- Init sniper ---
from sniper import ContextSniper
sniper = ContextSniper(embedding_model=config.EMBEDDING_MODEL)

print(f'\n✓ Ready')
print(f'  chats:  {config.CHATS_DIR}')
print(f'  db:     {config.DB_DIR}')
print(f'  model:  {config.EMBEDDING_MODEL}')
print(f'  mode:   {config.THOUGHT_MODE}')
print(f'  stats:  {sniper.stats()}')

## 1. Сброс базы (опционально)

Нужен когда меняются правила парсинга или хочешь начать с чистого листа.

In [ ]:
# @title 🗑 Reset DB { display-mode: "form" }
# @markdown Удаляет коллекцию текущей модели + манифест. Следующий ингест начнётся с нуля.
DROP_MANIFEST = True  # @param {type:"boolean"}
DROP_ALL_COLLECTIONS = False  # @param {type:"boolean"}
CONFIRM = False  # @param {type:"boolean"}

if not CONFIRM:
    print('⚠️ Поставь галку CONFIRM и перезапусти ячейку, чтобы подтвердить сброс.')
else:
    cleared = sniper.reset(drop_manifest=DROP_MANIFEST, drop_all_collections=DROP_ALL_COLLECTIONS)
    print(f'✓ Reset: {cleared}')
    print(f'  stats: {sniper.stats()}')

## 2. Ингест

Идемпотентно — повторные запуски обрабатывают только новые/изменённые файлы.
Первый раз модель скачается (~2 ГБ), дальше из кэша.

In [ ]:
# @title 📥 Ingest { display-mode: "form" }
SHOW_PROGRESS = True  # @param {type:"boolean"}

import time
t0 = time.time()
stats = sniper.ingest(config.CHATS_DIR, show_progress=SHOW_PROGRESS)
elapsed = time.time() - t0

print(f'\n=== DONE in {elapsed/60:.1f} min ===')
for k, v in stats.items():
    print(f'  {k}: {v}')
print(f'\n  {sniper.stats()}')

## 3. Поиск контекста

In [ ]:
# @title 🔍 Search { display-mode: "form" }
QUERY = "как я настраивал ChromaDB на Colab"  # @param {type:"string"}
TOP_K = 5  # @param {type:"slider", min:1, max:20, step:1}
SNIPPET_CHARS = 500  # @param {type:"slider", min:100, max:2000, step:100}

res = sniper.search_context(QUERY, top_k=TOP_K)
for i, (doc, meta, dist) in enumerate(zip(
    res['documents'][0], res['metadatas'][0], res['distances'][0]
)):
    print(f'--- #{i+1} | chat={meta["chat_id"]} | dist={dist:.3f} ---')
    print(doc[:SNIPPET_CHARS])
    print()

## 4. Супер-промпт

`MAIN_PROMPT` — твоя задача для LLM.
`META_QUERY` — по чему искать контекст (оставь пустым = искать по MAIN_PROMPT).

In [ ]:
# @title ✨ Generate Super-Prompt { display-mode: "form" }
MAIN_PROMPT = "Напиши обёртку над ингестом, которая логирует прогресс по файлам."  # @param {type:"string"}
META_QUERY = "ингест файлов без расширения, прогресс-логирование"  # @param {type:"string"}
TOP_K = 5  # @param {type:"slider", min:1, max:20, step:1}
MAX_DIST = 0.45  # @param {type:"slider", min:0.1, max:1.0, step:0.05}
SAVE_TO_FILE = True  # @param {type:"boolean"}

meta = META_QUERY or None
if SAVE_TO_FILE:
    path = sniper.save_super_prompt(
        main_prompt=MAIN_PROMPT,
        meta_query=meta,
        top_k=TOP_K,
        max_dist=MAX_DIST,
    )
    print(f'✓ Saved: {path}\n')
    print(path.read_text(encoding='utf-8'))
else:
    prompt = sniper.generate_super_prompt(
        main_prompt=MAIN_PROMPT,
        meta_query=meta,
        top_k=TOP_K,
        max_dist=MAX_DIST,
    )
    print(prompt)

## 5. Тегирование чатов (OpenRouter + DeepSeek V4 Flash)

Требует API-ключ. Добавь его в Colab Secrets (`🔑` слева) как `OPENROUTER_API_KEY`.

In [ ]:
# @title 🏷 Tag Chats { display-mode: "form" }
LLM_MODEL = "deepseek/deepseek-v4-flash"  # @param ["deepseek/deepseek-v4-flash", "deepseek/deepseek-v4-pro", "google/gemini-2.5-flash", "anthropic/claude-haiku-4"]
ONLY_UNTAGGED = True  # @param {type:"boolean"}

import requests
from google.colab import userdata
from functools import partial

def openrouter_call(prompt: str, api_key: str, model: str) -> str:
    resp = requests.post(
        'https://openrouter.ai/api/v1/chat/completions',
        headers={'Authorization': f'Bearer {api_key}'},
        json={'model': model, 'messages': [{'role': 'user', 'content': prompt}]},
    )
    resp.raise_for_status()
    return resp.json()['choices'][0]['message']['content']

llm = partial(openrouter_call, api_key=userdata.get('OPENROUTER_API_KEY'), model=LLM_MODEL)

results = sniper.tag_chats(llm, only_untagged=ONLY_UNTAGGED)
for r in results:
    print(f'{r.chat_id}: {r.tags}')

## 6. Сравнение моделей (опционально)

Каждая модель строит свою коллекцию — первый прогон по новой модели делает полный ингест.
**Внимание:** ×3 по времени и месту на Drive.

In [ ]:
# @title 🔬 Compare Models { display-mode: "form" }
QUERY = "как я настраивал ChromaDB на Colab"  # @param {type:"string"}
USE_BGE = True  # @param {type:"boolean"}
USE_QWEN = True  # @param {type:"boolean"}
USE_E5 = True  # @param {type:"boolean"}
RUN_INGEST = False  # @param {type:"boolean"}

MODELS = []
if USE_BGE: MODELS.append('bge-m3')
if USE_QWEN: MODELS.append('qwen3')
if USE_E5: MODELS.append('e5-instruct')

snipers = {}
for m in MODELS:
    s = ContextSniper(embedding_model=m)
    if RUN_INGEST:
        print(f'=== ingest with {m} ===')
        print(s.ingest(config.CHATS_DIR))
    snipers[m] = s

for m, s in snipers.items():
    print(f'\n========== {m} ==========')
    try:
        res = s.search_context(QUERY, top_k=3)
        for doc, meta, dist in zip(
            res['documents'][0], res['metadatas'][0], res['distances'][0]
        ):
            print(f'[{meta["chat_id"]}] dist={dist:.3f}  {doc[:200]}')
    except Exception as e:
        print(f'  Error: {e} (возможно коллекция пустая, включи RUN_INGEST)')